# Parakeet Live - realtime English speech-to-text on a free Colab GPU

Runs **NVIDIA Parakeet TDT 0.6B v2** (English) behind an HTTPS tunnel so your browser
mic can reach it. More accurate than Whisper on English, and much faster.

How it works: it waits for each pause in your speech, then transcribes the whole
sentence at once (that keeps Parakeet's accuracy). While you are still talking it
shows a grey live preview.

**Before running anything:**
1. If the Whisper notebook is still connected, close it first
   (**Runtime -> Disconnect and delete runtime** in that tab). The free tier usually
   gives you only one GPU at a time.
2. Here: **Runtime -> Change runtime type -> T4 GPU -> Save**.

Then run every cell in order. The last cell prints a `https://...trycloudflare.com`
link - open it, press **Start**, allow the mic.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Must print a GPU (e.g. Tesla T4). If not: Runtime -> Change runtime type -> T4 GPU.

## 2. Install

Uses `onnx-asr` with the GPU build of onnxruntime. The CPU-only onnxruntime is removed
first: if both are installed, the CPU one can win and the GPU is silently not used.

In [ ]:
!pip uninstall -y -q onnxruntime
!pip install -q "onnx-asr[gpu,hub]" fastapi uvicorn websockets
import onnxruntime as ort
print("onnxruntime", ort.__version__, "| providers:", ort.get_available_providers())

## 3. Get the code

In [ ]:
import os, subprocess

REPO = "https://github.com/kunalost/whisper.git"
DIR = "/content/whisper"
if not os.path.isdir(DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO, DIR], check=True)
else:
    subprocess.run(["git", "-C", DIR, "pull"], check=True)
print(sorted(f for f in os.listdir(DIR) if f.startswith("parakeet")))

## 4. Install cloudflared (the tunnel)

Colab has no public port, so cloudflared hands back a public **https** link.
It must be https: browsers block the microphone on plain http.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 5. Start the server

First run downloads the model (about 2.5 GB) - usually under a minute on Colab.
The last line must say **`"device": "GPU (CUDA)"`**. If it says `CPU`, see Troubleshooting.

In [ ]:
import subprocess, time, urllib.request, json

LOG = "/tmp/parakeet.log"
log = open(LOG, "w")
server = subprocess.Popen(
    ["python", "parakeet_server.py", "--device", "cuda", "--port", "8000"],
    cwd=DIR, stdout=log, stderr=subprocess.STDOUT,
)
print("server pid", server.pid, "- loading model, please wait...")

for i in range(90):
    if server.poll() is not None:
        print("SERVER EXITED. Log tail:")
        print(open(LOG).read()[-3000:])
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as r:
            health = json.loads(r.read())
            print("READY:", health)
            if health["device"] != "GPU (CUDA)":
                print("!! Running on CPU, not the GPU. See Troubleshooting below.")
            break
    except Exception:
        time.sleep(10)
else:
    print("timed out. Log tail:")
    print(open(LOG).read()[-3000:])

## 6. Open the tunnel

Runs until you stop it - that is how it holds the link open. Find the boxed
`https://...trycloudflare.com` link below, open it in a new tab, press **Start**,
allow the microphone.

The link is public while this runs. Do not share it, and stop this cell when done.

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8000 --no-autoupdate

## Troubleshooting

| Symptom | Fix |
|---|---|
| Cell 5 says **CPU** instead of `GPU (CUDA)` | onnxruntime could not load the CUDA libraries. Run `!pip install -q "onnxruntime-gpu[cuda,cudnn]"`, then `!pkill -f parakeet_server.py`, then cell 5 again. |
| `SERVER EXITED` in cell 5 | Read the log tail it prints and send it over. |
| Page says *server not reachable* | Cell 5 is not running or failed. Re-run cell 5, then cell 6 (new link). |
| Mic permission never asked | You opened an `http://` link. Use `https://`. |
| Level bar does not move when you talk | Wrong microphone. Click the mic icon in the address bar and pick the right one, then Stop/Start. |
| Text only appears after you pause | That is by design: final text comes at each pause. Grey text is the live preview. |
| Everything stops | Session hit its time/idle limit. Re-run the notebook. |

To change settings (for example a longer pause before a sentence is finalized),
stop cell 6, run `!pkill -f parakeet_server.py`, add options to the command in cell 5
such as `"--end-silence", "0.8"`, then run cells 5 and 6 again.
Run `!cd /content/whisper && python parakeet_server.py --help` to see all options.